# Solar filament segmentation: the whole pipeline

This notebook walks through our solution from the raw H-alpha image to the submission file and
then regenerates the submission **from the weights stored in this repository** (no retraining).

| Section | What it shows | Hardware |
|---|---|---|
| 1. Setup | packages, data, stored weights (`weights/manifest.json`) | CPU or GPU |
| 2. Data | counts, one observation with every expert's filaments | CPU or GPU |
| 3. Preprocessing | 2048 → 1024 resize, on-disk normalization, soft (annotator-mean) target | CPU or GPU |
| 4. Model and training | architecture, training settings and commands (training is not re-run) | CPU or GPU |
| 5. Inference | the five fold models and test-time augmentation on one test image | CPU or GPU |
| 6. Post-processing | probability map → separate filament instances, step by step | CPU or GPU |
| 7. Validation | cross-validated scores and how PQ is computed | CPU or GPU |
| 8. Full reproduction | all 180 test images → `submissions/reproduced.csv`, compared with the submitted file | **GPU** |

Sections 1–7 use one image each and run on a CPU in a few minutes. Section 8 runs only when a CUDA GPU is
available (8 GB is enough).

**Requirements:** Python 3.12 and the packages in `requirements.txt`. Place the competition data at
`data/MAGFiLO_1.0_Kaggle_2026/` (as unzipped from Kaggle). Section 8 needs only the test images.

## 1. Setup

The project code lives in `src/`; the notebook imports it instead of re-implementing it, so what is shown
here is exactly what produced the submission.

In [ ]:
import json, sys, time
from pathlib import Path

# Make src/ importable whether the notebook is started from the repository root or from notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import sklearn
import torch

from common import (FULL_SIZE, TEST_IMG_DIR, TRAIN_IMG_DIR, instance_masks, load_annotations, normalize_disk,
                    read_gray, union_mask)

plt.rcParams["figure.dpi"] = 72  # small figures keep the notebook light

# Illustrations (sections 1-7) run on CPU too; the full reproduction (section 8) needs a GPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_FULL_REPRODUCTION = torch.cuda.is_available()

print("python", sys.version.split()[0], "| torch", torch.__version__, "| smp", smp.__version__,
      "| scikit-learn", sklearn.__version__, "| opencv", cv2.__version__, "| numpy", np.__version__)
print("device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("full reproduction (section 8):", RUN_FULL_REPRODUCTION)

In [ ]:
# The stored solution: model list, test-time augmentation and post-processing (written by src/export_weights.py).
import os
WEIGHTS = ROOT / "weights"
# manifest.json = the final configuration; manifest_B.json = our second selected submission
# (the same model with a second training seed). Choose with the FIL_MANIFEST environment variable.
MANIFEST_FILE = os.environ.get("FIL_MANIFEST", "manifest.json")
manifest = json.loads((WEIGHTS / MANIFEST_FILE).read_text())
runs = manifest["models"]
print(json.dumps(manifest, indent=2))

# Check that the competition data is where src/common.py expects it.
test_stems = sorted(p.stem for p in TEST_IMG_DIR.glob("*.jpeg"))
assert len(test_stems) == 180, f"competition test images not found in {TEST_IMG_DIR}"

## 2. Data

Full-disk H-alpha images (2048 × 2048, 8-bit grayscale JPEG) from the six GONG sites. The file name is
`YYYYMMDDhhmmss` plus a site code (e.g. `Bh` = Big Bear, H-alpha). Every filament is annotated as one
polygon (COCO format). Many observations were annotated by two or three experts independently; each such
**annotation set** is scored separately by the leaderboard.

In [ ]:
index, polygons = load_annotations()  # one row per annotation set (expert x observation), with its CV fold
sets_per_obs = index.groupby("stem").size()
print(f"training observations:   {index['stem'].nunique()}")
print(f"annotation sets:         {len(index)}  (observations with 1/2/3 sets: "
      f"{sets_per_obs.value_counts().sort_index().to_dict()})")
print(f"annotated filaments:     {index['n_filaments'].sum()}  (all annotation sets)")
print(f"test images:             {len(test_stems)}")

SITES = {"B": "Big Bear", "C": "Cerro Tololo", "L": "Learmonth", "M": "Mauna Loa", "T": "El Teide", "U": "Udaipur"}
site = index.drop_duplicates("stem")["stem"].str[-2].map(SITES)
print("training observations per site:", site.value_counts().to_dict())

One observation annotated by three experts. The polygons are rasterized with `pycocotools`, the routine the
host uses to encode masks, and each expert's filaments are outlined in a different color.

In [ ]:
TRAIN_STEM = "20160810234034Lh"   # three annotation sets; their agreement (PQ about 0.35) is typical
sets = index[index["stem"] == TRAIN_STEM]
display(sets)

raw = read_gray(TRAIN_IMG_DIR / f"{TRAIN_STEM}.jpeg")                           # 2048 x 2048 uint8
expert_masks = {i: instance_masks(polygons[i]) for i in sets["image_id"]}       # one binary mask per filament
COLORS = [(255, 60, 60), (60, 200, 255), (255, 220, 0)]


def draw_outlines(rgb, masks, color, thickness):
    # Outline every binary mask on an RGB image (in place).
    for m in masks:
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, contours, -1, color, thickness)
    return rgb


def window(center_yx, half, size):
    # Square crop around a point, kept inside the image.
    y, x = (int(np.clip(c, half, size - half)) for c in center_yx)
    return slice(y - half, y + half), slice(x - half, x + half)


# zoom on the largest filament of the first expert to see where the experts disagree
largest = max(expert_masks[sets["image_id"].iloc[0]], key=lambda m: m.sum())
zoom = window(np.argwhere(largest).mean(0), 300, FULL_SIZE)

full = cv2.cvtColor(raw, cv2.COLOR_GRAY2RGB)
crop = cv2.cvtColor(raw[zoom], cv2.COLOR_GRAY2RGB)
for color, masks in zip(COLORS, expert_masks.values()):
    draw_outlines(full, masks, color, 10)
    draw_outlines(crop, [m[zoom] for m in masks], color, 3)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(full); ax[0].set_title(f"{TRAIN_STEM}: {len(sets)} experts")
ax[1].imshow(crop); ax[1].set_title("zoom")
handles = [plt.Line2D([], [], color=np.array(c) / 255, lw=3) for c in COLORS]
labels = [f"expert {i.split('-')[0]}: {len(m)} filaments" for i, m in expert_masks.items()]
ax[1].legend(handles, labels, loc="lower left", fontsize=8)
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

**The experts disagree a lot.** Scoring one expert against another with the leaderboard metric (Panoptic
Quality, section 7) shows how much. Over all 296 observations with two or three annotation sets
(1,196 ordered pairs), expert-vs-expert agreement is only **PQ ≈ 0.34** (Dice ≈ 0.61), computed with
`src/annotator_agreement.py`. This is the reference for the model's scores; the pairs of this observation:

In [ ]:
from itertools import combinations
from annotator_agreement import to_label_map
from metrics import pq_stats

# PQ of expert a's filaments (as prediction) against expert b's (as ground truth); symmetric in a and b.
for a, b in combinations(expert_masks, 2):
    s = pq_stats(to_label_map(expert_masks[a]), expert_masks[b])
    print(f"expert {a.split('-')[0]} vs {b.split('-')[0]}: PQ {s.pq:.2f}  (matched {s.tp}, unmatched {s.fp} + {s.fn})")

## 3. Preprocessing

* **Resize 2048 → 1024** with `cv2.INTER_AREA` (`src/prepare.py`). The network works at 1024 px to fit an 8 GB
  GPU; area averaging keeps thin dark filaments visible instead of skipping pixels.
* **On-disk normalization** (`common.normalize_disk`): subtract the median and divide by the standard deviation of
  the solar-disk pixels only (brighter than 20), clip to [−6, 6]. The GONG sites differ in exposure and contrast;
  using disk statistics (not the dark sky) makes the inputs comparable.
* **Soft target** (`prepare.soft_target`): the training target of an observation is the per-pixel *mean* of its
  experts' filament masks, i.e. the fraction of experts who marked the pixel (0, 1/3, 1/2, 2/3, 1).

In [ ]:
from prepare import soft_target

small = cv2.resize(raw, (1024, 1024), interpolation=cv2.INTER_AREA)  # same call as prepare.py
x = normalize_disk(small)                                             # float32 network input
disk = small > 20
print(f"raw {raw.shape} {raw.dtype} | resized {small.shape} | normalized {x.dtype}: "
      f"disk median {np.median(x[disk]):.2f}, disk std {x[disk].std():.2f}, range [{x.min():.1f}, {x.max():.1f}]")

# soft target at the network resolution, built from each expert's union mask
soft = soft_target([union_mask(polygons[i], 1024) for i in sets["image_id"]]) / 255.0
print("soft target values:", np.unique(soft.round(3)))

zoom1024 = tuple(slice(s.start // 2, s.stop // 2) for s in zoom)  # the zoom of section 2 at 1024 px
fig, ax = plt.subplots(1, 5, figsize=(16, 3.4))
ax[0].imshow(raw, cmap="gray"); ax[0].set_title("raw 2048 px")
ax[1].imshow(small, cmap="gray"); ax[1].set_title("INTER_AREA 1024 px")
im = ax[2].imshow(x, cmap="gray", vmin=-6, vmax=6); ax[2].set_title("normalize_disk")
plt.colorbar(im, ax=ax[2], fraction=0.046)
im = ax[3].imshow(soft, cmap="magma", vmin=0, vmax=1); ax[3].set_title("soft target")
plt.colorbar(im, ax=ax[3], fraction=0.046)
ax[4].imshow(soft[zoom1024], cmap="magma", vmin=0, vmax=1); ax[4].set_title("soft target (zoom)")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

## 4. Model and training

**Architecture.** A U-Net (`segmentation_models_pytorch`) with an ImageNet-pretrained EfficientNetV2-S encoder
(`timm`). The input has one channel (the grayscale image; the pretrained first convolution sums its RGB filters)
and the output is one per-pixel filament logit. The settings below are read from each stored run's
`config.json`, which is the argument list `src/train.py` was called with.

**Training** (`src/train.py`), in short:
* **Soft targets** (`--soft`): one sample per observation with the annotator-mean target of section 3 instead of
  one sample per expert. The model learns how many experts would mark a pixel, which suits a metric that scores
  every expert separately. Loss: binary cross-entropy + Dice (the Dice semimetric form, which is minimized at
  prediction = soft target).
* **Weight EMA** (`--ema`): an exponential moving average of the weights (decay 0.998 per step) is validated and
  saved; it is smoother than the raw weights.
* **Grouped 5-fold cross-validation**: folds are drawn by observation (`common.make_folds`), so the same image
  labelled by different experts never appears in both training and validation. Fold *k*'s model is validated on
  fold *k* (PQ and Dice per annotation set, every 5 epochs) and the checkpoint with the best (Dice + PQ) / 2 is kept.
* AdamW (lr 3e-4, weight decay 1e-4), one epoch of linear warm-up then cosine decay, mixed precision, random
  768 px crops of the 1024 px images, batch 4, 65 epochs. Augmentation: rotation by any angle, scaling 0.9–1.1,
  flips (the disk has no preferred orientation), brightness/contrast, gamma, blur, noise.
* All five fold models are used at test time (ensemble). Their out-of-fold predictions on the 707 training images
  were used to tune the post-processing (section 6) and to estimate the score (section 7).

Training is **not** re-run here (about 30 minutes per fold on an RTX 5060 8 GB). The commands are:

In [ ]:
configs = [json.loads((WEIGHTS / r / "config.json").read_text()) for r in runs]
display(pd.DataFrame(configs).set_index("name"))


def train_command(cfg):
    # Rebuild the train.py command line from a stored config.json (= the parsed arguments).
    args = [f"--name {cfg['name']}"]
    for key, value in cfg.items():
        if key == "name" or value is None or value is False:
            continue                      # options that were not set keep their defaults
        if key == "spine_weight" and not cfg.get("spine"):
            continue                      # only used together with --spine
        flag = "--" + key.replace("_", "-")
        args.append(flag if value is True else f"{flag} {value}")
    return "python src/train.py " + " ".join(args)


print("python src/prepare.py   # once: resized images, masks and soft targets in data/cache/")
for cfg in configs:
    print(train_command(cfg))

In [ ]:
from model import load_trained, tta_views

model, size = load_trained(WEIGHTS / runs[0], device="cpu")  # fp16 weights are loaded into a float32 model
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__} + {configs[0]['encoder']}: {n_params / 1e6:.1f} M parameters, "
      f"input 1 x {size} x {size}, output 1 x {size} x {size} (filament logit)")
print("encoder feature channels per scale:", model.encoder.out_channels)
del model

## 5. Inference on one test image

For every stored fold model: resize the test image to the model's resolution, normalize it, and predict the
filament probability (sigmoid of the logit) with **test-time augmentation**. The `"d4"` mode uses the 8
symmetries of the square (the 4 flips of the image and of its transpose, i.e. all 90° rotations and
reflections): each view is predicted, transformed back and the 8 maps are averaged. A filament looks the
same in any orientation, so the views should agree; averaging removes orientation-specific noise (about
+0.003 PQ out-of-fold compared with 4 flips). The five fold models are then averaged.

`infer.predict_maps` (section 8) does exactly this on the GPU with mixed precision and stores the averaged map as
an 8-bit PNG (probability × 255) at 1024 px; it is upsampled to 2048 px before post-processing.

In [ ]:
from postprocess import upsample_prob

TEST_STEM = "20150602122934Ch"   # a test image on which every post-processing step of section 6 is visible
test_raw = read_gray(TEST_IMG_DIR / f"{TEST_STEM}.jpeg")
views = tta_views(manifest["tta"])
print(f"TTA {manifest['tta']!r}: {len(views)} views (transpose?, flipped axes):", views)


@torch.no_grad()
def predict_views(model, x):
    # Probability map of every TTA view, mapped back to the original orientation (views x H x W).
    # Same steps as model.predict_prob, in float32 so that it also runs on a CPU.
    maps = []
    for transpose, dims in views:
        xi = x.transpose(2, 3) if transpose else x
        xi = torch.flip(xi, dims) if dims else xi
        p = torch.sigmoid(model(xi.contiguous())[:, :1])
        p = torch.flip(p, dims) if dims else p
        maps.append(p.transpose(2, 3) if transpose else p)
    return torch.cat(maps)[:, 0].float().cpu().numpy()


sizes = {cfg["name"]: int(cfg["size"]) for cfg in configs}
out_size = max(sizes.values())        # maps are averaged at the finest model resolution
per_model, first_views = {}, None
t0 = time.time()
for run in runs:
    model, size = load_trained(WEIGHTS / run, device=device)
    img = cv2.resize(test_raw, (size, size), interpolation=cv2.INTER_AREA) if size != FULL_SIZE else test_raw
    x = torch.from_numpy(normalize_disk(img))[None, None].to(device)
    maps = predict_views(model, x)
    p = maps.mean(0)                  # D4 average of this model
    if size != out_size:              # (predict_maps uses the equivalent bilinear torch interpolation)
        p = cv2.resize(p, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    per_model[run] = p
    if first_views is None:
        first_views = maps            # kept to show how much the TTA views differ
    del model
print(f"{len(runs)} models x {len(views)} views on {device}: {time.time() - t0:.0f} s")

prob_avg = np.mean(list(per_model.values()), axis=0)
# As in the submission: store as 8-bit, then upsample to the 2048 px submission resolution.
prob8 = np.round(prob_avg * 255).astype(np.uint8)
prob = upsample_prob(prob8.astype(np.float32) / 255.0, FULL_SIZE)

# How much do views and models disagree? Pixels whose filament/background decision (> 0.5) differs from the
# final ensemble, relative to the ensemble's filament area.
fg = prob_avg > 0.5
def differs(p):
    return ((p > 0.5) != fg).sum() / fg.sum()
print(f"{runs[0]}: one view {differs(first_views[0]):.1%}, D4 average {differs(per_model[runs[0]]):.1%}")
print("D4 average of each model:", {run: f"{differs(p):.1%}" for run, p in per_model.items()})

In [ ]:
# Zoom on the region with the most predicted filament pixels.
half = 192
density = cv2.blur((prob_avg > 0.5).astype(np.float32), (201, 201))
zoom_t = window(np.unravel_index(density.argmax(), density.shape), half, out_size)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(test_raw, cmap="gray"); ax[0].set_title(f"test image {TEST_STEM}")
ax[1].imshow(prob_avg, cmap="magma", vmin=0, vmax=1)
ax[1].set_title(f"ensemble probability ({len(runs)} models x {len(views)} views)")
s = out_size / FULL_SIZE                      # draw the zoom window on the 2048 px image
y0, x0 = zoom_t[0].start / s, zoom_t[1].start / s
ax[0].add_patch(plt.Rectangle((x0, y0), 2 * half / s, 2 * half / s, fill=False, ec="cyan", lw=2))
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

panels = [("image", cv2.resize(test_raw, (out_size, out_size), interpolation=cv2.INTER_AREA)[zoom_t], "gray")]
panels += [(run, p[zoom_t], "magma") for run, p in per_model.items()]
panels += [("ensemble", prob_avg[zoom_t], "magma"), ("D4 view spread (std), " + runs[0], first_views.std(0)[zoom_t], "viridis")]
nrows = -(-len(panels) // 4)                  # 4 panels per row, as many rows as needed
fig, ax = plt.subplots(nrows, 4, figsize=(12, 3.2 * nrows))
for a in ax.ravel():
    a.axis("off")
for a, (title, im, cmap) in zip(ax.ravel(), panels):
    a.imshow(im, cmap=cmap, vmin=0, vmax=1 if cmap == "magma" else None)
    a.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

## 6. Post-processing: probability map → filament instances

The network says *where* filament pixels are; the leaderboard scores *instances* (one mask per filament), so
the map must be cut into filaments (`src/postprocess.py`). The parameters come from the manifest; they were
tuned on the out-of-fold maps of the 707 training images.

1. **Threshold** the probability at `threshold` (0.5).
2. **Connected components** (8-connected) of the thresholded map = candidate fragments.
3. **Hysteresis grouping** (`group_low` 0.2, `merge_px` 8): two fragments become one filament when they are
   connected through pixels with probability > 0.2 (gaps up to 8 px bridged). A filament often has a faint middle
   section that drops below 0.5; the faint ridge tells which fragments belong together, which is more selective
   than joining any two fragments that happen to be close. Each instance keeps only its pixels above 0.5.
4. **Learned filter**: a gradient-boosted regressor (`weights/instance_scorer.joblib`, bagged scikit-learn
   `HistGradientBoostingRegressor`) predicts each candidate's *matched IoU* (its IoU with an annotated filament if
   that exceeds 0.5, else 0, averaged over the experts) from 23 descriptors (`src/instance_features.py`):
   probability statistics, shape, stability across nearby thresholds and image context. It was trained on the
   out-of-fold candidates of the training images. Candidates smaller than `min_area` (50 px) or with a predicted
   matched IoU not above `min_score` (0.2) are dropped.

**Why 0.2?** Reporting an instance always adds 0.5 to the PQ denominator (a true positive replaces a false
negative: +1 − 0.5; a false positive: +0.5) and adds its matched IoU *u* to the numerator. So it raises
PQ = S / D exactly when (S + u) / (D + 0.5) > S / D, i.e. when **u > PQ / 2**. With PQ ≈ 0.45 the break-even is
about 0.2.

In [ ]:
from make_submission import load_params
from postprocess import instance_confidence, label_components, predict_matched_iou, to_instances

params = load_params(str(WEIGHTS / MANIFEST_FILE))
print(params)
t0 = time.time()

# Steps 1-2: threshold and connected components (no grouping).
binary = prob > params.threshold
fragments = label_components(prob, params.threshold)
# Step 3: fragments grouped (hysteresis or distance merging, as configured).
grouped = label_components(prob, params.threshold, params.merge_px, params.group_low)
print(f"{binary.sum()} pixels above {params.threshold}: {fragments.max()} fragments -> {grouped.max()} candidates")

# Which fragments were joined: count the distinct fragments inside every candidate.
pairs = pd.DataFrame({"candidate": grouped[binary], "fragment": fragments[binary]}).drop_duplicates()
n_frag = pairs.groupby("candidate").size().reindex(range(int(grouped.max()) + 1), fill_value=0).to_numpy()
print("candidates made of several fragments:", {int(k): int(n_frag[k]) for k in np.flatnonzero(n_frag > 1)})

In [ ]:
# Zoom on the candidate joined from the most fragments.
area = np.bincount(grouped.ravel())
joined = np.flatnonzero(n_frag > 1)
if len(joined):
    k = joined[np.argmax(n_frag[joined])]
    zoom_g = window(np.argwhere(grouped == k).mean(0), 160, FULL_SIZE)
    palette = np.random.default_rng(0).integers(60, 256, (int(fragments.max()) + 1, 3)).astype(np.uint8)
    palette[0] = 0                                    # background stays black

    fig, ax = plt.subplots(1, 4, figsize=(14, 3.8))
    ax[0].imshow(prob[zoom_g], cmap="magma", vmin=0, vmax=1); ax[0].set_title("probability")
    ax[1].imshow(palette[fragments[zoom_g]]); ax[1].set_title(f"fragments (> {params.threshold})")
    ax[2].imshow(prob[zoom_g] > params.group_low, cmap="gray"); ax[2].set_title(f"grouping mask (> {params.group_low})")
    ax[3].imshow(palette[grouped[zoom_g]]); ax[3].set_title(f"grouped: {n_frag[k]} fragments -> 1")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("no fragments were joined on this image")

In [ ]:
# Step 4: the filter, written out exactly as in postprocess.to_instances.
max_prob, mean_prob = instance_confidence(grouped, prob, area)
keep = (area >= max(params.min_area, 1)) & (max_prob >= params.min_conf) & (mean_prob >= params.min_meanp)
keep[0] = False                                       # label 0 is the background
score = np.full(area.size, np.nan)
if params.scorer:                                     # learned filter (predicted matched IoU)
    score = predict_matched_iou(grouped, prob, params)
    keep &= score > params.min_score

table = pd.DataFrame({"area_px": area, "fragments": n_frag, "peak_prob": max_prob, "mean_prob": mean_prob,
                      "pred_matched_iou": score, "kept": keep}).iloc[1:]
print(f"{keep.sum()} kept, {(table.area_px < params.min_area).sum()} too small (< {params.min_area} px), "
      f"{((table.area_px >= params.min_area) & ~table.kept).sum()} dropped by the filter")
display(table[table.area_px >= params.min_area].sort_values("pred_matched_iou", ascending=False).round(3))

# Step 5: the library call used for the submission gives the same instances.
final = to_instances(prob, params)
assert final.max() == keep.sum() and np.array_equal(final > 0, keep[grouped])
print(f"post-processing of one image: {time.time() - t0:.1f} s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
vis = cv2.cvtColor(test_raw, cv2.COLOR_GRAY2RGB)
for k in range(1, area.size):
    mask = grouped == k
    if area[k] < params.min_area:
        draw_outlines(vis, [mask], (150, 150, 150), 8)   # too small to be scored
        continue
    draw_outlines(vis, [mask], (40, 220, 40) if keep[k] else (255, 50, 50), 8)
    if params.scorer:                                    # predicted matched IoU next to the candidate
        y, x = np.argwhere(mask).mean(0)
        ax[0].text(x + 40, y, f"{score[k]:.2f}", color="yellow", fontsize=8)
ax[0].imshow(vis); ax[0].set_title("candidates: kept (green), dropped (red), too small (gray)")

palette = np.random.default_rng(1).integers(60, 256, (int(final.max()) + 1, 3)).astype(np.uint8)
overlay = cv2.cvtColor(test_raw, cv2.COLOR_GRAY2RGB)
overlay[final > 0] = palette[final[final > 0]]
ax[1].imshow(overlay); ax[1].set_title(f"final: {final.max()} filament instances")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

# Each instance becomes one submission row: COCO compressed RLE of its 2048 x 2048 mask.
from make_submission import encode
print(f"filament_id,segmentation_rle\n{TEST_STEM}_1,{encode(final == 1)[:70]}...")

## 7. Validation

**Panoptic Quality** (leaderboard metric, `src/metrics.py`, same definition as the host's evaluator). For every
annotation set (one expert's filaments on one image), a predicted instance and an annotated filament match when
their IoU exceeds 0.5 (such a match is unique). Matched pairs are true positives (TP), unmatched predictions false
positives (FP), unmatched filaments false negatives (FN):

$$\mathrm{PQ} = \frac{\sum_{(p, g) \in TP} \mathrm{IoU}(p, g)}{|TP| + \tfrac12 |FP| + \tfrac12 |FN|}$$

The sums are **pooled** over all (image, expert) pairs before dividing, so an observation with three experts is
scored three times against the same prediction. **Dice** is computed on the union masks of every annotation set
and averaged.

**Cross-validated estimate.** Every training image is predicted by the fold model that did not see it
(out-of-fold, 707 images). The learned filter is cross-fitted (each fold scored by a filter trained on the other
four folds), and all metrics are computed at 2048 px. For reference, two experts agree with each other at
PQ ≈ 0.34 (section 2): the model can score higher than a single expert because it predicts the consensus.

In [ ]:
pd.Series(manifest["validation"], name="out-of-fold (707 training images)").to_frame().round(4)

## 8. Full reproduction of the submission (GPU)

Regenerates the submission from the stored weights and the 180 test images, with the same functions as the
command-line scripts:

1. resize the test images to the models' resolution (`prepare.cache_images`, written to `data/cache/`);
2. probability maps of all models with the manifest's TTA (`infer.predict_maps`, written to `preds/reproduced/test/`);
3. instances and the submission file (`make_submission.build_submission`, `validate`) → `submissions/reproduced.csv`;
4. comparison with the file submitted to Kaggle (SHA-256 fingerprint and row count stored in the manifest).

The cells are skipped when no CUDA GPU is available.

In [ ]:
if RUN_FULL_REPRODUCTION:
    from prepare import cache_images
    for size in sorted(set(sizes.values())):   # sizes: model resolutions read in section 5
        cache_images(size, include_train=False)
else:
    print("No CUDA GPU: section 8 is skipped.")

In [ ]:
if RUN_FULL_REPRODUCTION:
    from common import PRED_DIR
    from infer import predict_maps, split_stems

    stems = split_stems("test")
    prob_dir = PRED_DIR / "reproduced" / "test"
    seconds = predict_maps(runs, stems, prob_dir, weights_root=WEIGHTS, tta=manifest["tta"])
    print(f"inference ({manifest['tta']}): {seconds / 60:.1f} min for {len(stems)} images")

In [ ]:
if RUN_FULL_REPRODUCTION:
    import hashlib
    from common import SUB_DIR
    from make_submission import build_submission, validate

    t0 = time.time()
    sub = build_submission(prob_dir, stems, params)   # params: post-processing from the manifest (section 6)
    validate(sub)
    SUB_DIR.mkdir(exist_ok=True)
    out = SUB_DIR / "reproduced.csv"
    sub.to_csv(out, index=False)
    print(f"post-processing: {(time.time() - t0) / 60:.1f} min | wrote {out}")
    display(sub.head())

GPU convolution algorithms can round a few borderline pixels differently between runs, so a few masks may
differ slightly from the submitted file; the number of filaments should match.

In [ ]:
if RUN_FULL_REPRODUCTION:
    sha = hashlib.sha256(out.read_bytes()).hexdigest()
    print(f"reproduced: {len(sub)} rows | submitted ({manifest['submitted_file']}): {manifest.get('submitted_rows')} rows")
    print("sha256 identical to the submitted file:", sha == manifest.get("submitted_sha256"))
    submitted = SUB_DIR / manifest["submitted_file"]
    if submitted.exists():                            # only on the machine that made the submission
        ref = pd.read_csv(submitted)
        print(f"rows identical to the submitted file: {len(sub.merge(ref))} / {len(ref)}")

## Summary: where each step lives

| Step | Code |
|---|---|
| annotations, polygon rasterization, grouped folds | `src/common.py` |
| resize 2048 → 1024, soft targets | `src/prepare.py` |
| normalization, augmentation, dataset | `src/common.py` (`normalize_disk`), `src/dataset.py` |
| U-Net + EfficientNetV2-S, D4 test-time augmentation | `src/model.py` |
| training (soft targets, EMA, per-fold validation) | `src/train.py` |
| probability maps (5 models × 8 views averaged) | `src/infer.py` |
| threshold → components → hysteresis grouping → learned filter | `src/postprocess.py`, `src/instance_features.py` |
| fitting the learned filter on out-of-fold maps | `src/instance_scorer.py` |
| PQ and Dice | `src/metrics.py` |
| submission file (COCO RLE) | `src/make_submission.py` |
| stored weights and manifest | `src/export_weights.py` → `weights/` |